In [1]:
import pandas as pd
from framework.utils import get_unseen_prefix_subset, prepare_datasets, generate_prefix_data, split_cases_temporal, split_cases, preprocess_log, get_event_log_statistics, encode_prefix_data, build_process_vocab, build_activity_vocab, count_unique_traces, filter_complete_cases


# Paper results

In [2]:
scenarios = ['scenario_1_A', 'scenario_1_B_75_unique',  'scenario_1_B_40_unique', 'scenario_1_B_20_unique', 'scenario_1_B']
gen_names = ['scenario_1_A_GEN', 'scenario_1_B_75_unique_GEN', 'scenario_1_B_40_unique_GEN', 'scenario_1_B_20_unique_GEN', 'scenario_1_B_GEN']
models = ['LSTM_PYTORCH', 'TRANSFORMER','MODEL4']

In [11]:
import pandas as pd
import numpy as np
from scipy import stats

results = []


models = ['LSTM_PYTORCH', 'TRANSFORMER', 'LEVEL3_XGB','PGT','TIMEAWARE']
scenarios = ['scenario_1_A', 'scenario_1_B_75_unique',  'scenario_1_B_40_unique', 'scenario_1_B_20_unique', 'scenario_1_B', 'bpi2020_2processes_massive_share']

for scenario in scenarios:
    row = {'Scenario': scenario}
    
    for model in models:
        try:
            df = pd.read_csv(f'results/{model}_{scenario}.csv')[['test_mae']]
            mae_mean = df['test_mae'].mean()
            mae_std = df['test_mae'].std()
            n = len(df)
            
            ci = stats.t.ppf(0.975, df=n-1) * (mae_std / np.sqrt(n)) if n > 1 else 0.0
            
            value = f"{mae_mean:.2f} (±{ci:.2f})"
            row[model] = value
        
        except Exception as e:
            row[model] = 'N/A'
    
    results.append(row)

results_df = pd.DataFrame(results)

In [12]:
results_df

,Scenario,LSTM_PYTORCH,TRANSFORMER,LEVEL3_XGB,PGT,TIMEAWARE
0,scenario_1_A,11.79 (±2.85),9.30 (±0.14),9.00 (±0.00),8.53 (±0.00),8.96 (±0.28)
1,scenario_1_B_75_unique,16.51 (±0.23),9.45 (±0.27),9.12 (±0.00),9.05 (±0.00),8.62 (±0.08)
2,scenario_1_B_40_unique,69.98 (±0.35),19.33 (±0.40),16.10 (±0.00),15.42 (±0.00),15.59 (±0.21)
3,scenario_1_B_20_unique,183.86 (±0.62),114.99 (±2.63),121.00 (±0.00),118.30 (±0.00),102.18 (±3.64)
4,scenario_1_B,77.65 (±82.24),46.85 (±0.52),49.60 (±0.00),47.41 (±0.00),45.03 (±0.36)
5,bpi2020_2processes_massive_share,N/A,2.50 (±0.01),2.82 (±0.00),2.45 (±0.00),2.49 (±0.00)


In [13]:
results_df

,Scenario,LSTM_PYTORCH,TRANSFORMER,LEVEL3_XGB,PGT,TIMEAWARE
0,scenario_1_A,11.79 (±2.85),9.30 (±0.14),9.00 (±0.00),8.53 (±0.00),8.96 (±0.28)
1,scenario_1_B_75_unique,16.51 (±0.23),9.45 (±0.27),9.12 (±0.00),9.05 (±0.00),8.62 (±0.08)
2,scenario_1_B_40_unique,69.98 (±0.35),19.33 (±0.40),16.10 (±0.00),15.42 (±0.00),15.59 (±0.21)
3,scenario_1_B_20_unique,183.86 (±0.62),114.99 (±2.63),121.00 (±0.00),118.30 (±0.00),102.18 (±3.64)
4,scenario_1_B,77.65 (±82.24),46.85 (±0.52),49.60 (±0.00),47.41 (±0.00),45.03 (±0.36)
5,bpi2020_2processes_massive_share,N/A,2.50 (±0.01),2.82 (±0.00),2.45 (±0.00),2.49 (±0.00)


In [14]:
results_df.to_latex()

'\\begin{tabular}{lllllll}\n\\toprule\n & Scenario & LSTM_PYTORCH & TRANSFORMER & LEVEL3_XGB & PGT & TIMEAWARE \\\\\n\\midrule\n0 & scenario_1_A & 11.79 (±2.85) & 9.30 (±0.14) & 9.00 (±0.00) & 8.53 (±0.00) & 8.96 (±0.28) \\\\\n1 & scenario_1_B_75_unique & 16.51 (±0.23) & 9.45 (±0.27) & 9.12 (±0.00) & 9.05 (±0.00) & 8.62 (±0.08) \\\\\n2 & scenario_1_B_40_unique & 69.98 (±0.35) & 19.33 (±0.40) & 16.10 (±0.00) & 15.42 (±0.00) & 15.59 (±0.21) \\\\\n3 & scenario_1_B_20_unique & 183.86 (±0.62) & 114.99 (±2.63) & 121.00 (±0.00) & 118.30 (±0.00) & 102.18 (±3.64) \\\\\n4 & scenario_1_B & 77.65 (±82.24) & 46.85 (±0.52) & 49.60 (±0.00) & 47.41 (±0.00) & 45.03 (±0.36) \\\\\n5 & bpi2020_2processes_massive_share & N/A & 2.50 (±0.01) & 2.82 (±0.00) & 2.45 (±0.00) & 2.49 (±0.00) \\\\\n\\bottomrule\n\\end{tabular}\n'

In [15]:
import pandas as pd
import numpy as np
from scipy import stats

results = []

models = ['LSTM_PYTORCH', 'TRANSFORMER', 'LEVEL3_XGB','PGT', 'TIMEAWARE']
scenarios = ['scenario_1_A', 'scenario_1_B_75_unique',  'scenario_1_B_40_unique', 'scenario_1_B_20_unique', 'scenario_1_B','bpi2020_2processes_massive_share']

for scenario in scenarios:
    row = {'Scenario': scenario}
    
    for model in models:
        try:
            df = pd.read_csv(f'results/{model}_{scenario}.csv')[['training_time_sec']]
            df['training_time_min'] = df['training_time_sec'] / 60 
            
            mean_min = df['training_time_min'].mean()
            std_min = df['training_time_min'].std()
            n = len(df)
            
            ci = stats.t.ppf(0.975, df=n-1) * (std_min / np.sqrt(n)) if n > 1 else 0.0
            
            value = f"{mean_min:.2f} (±{ci:.2f})"
            row[model] = value
        
        except Exception as e:
            row[model] = 'N/A'
    
    results.append(row)

results_df = pd.DataFrame(results)
results_df

,Scenario,LSTM_PYTORCH,TRANSFORMER,LEVEL3_XGB,PGT,TIMEAWARE
0,scenario_1_A,2.79 (±0.97),2.40 (±0.47),N/A,490.02 (±0.00),73.38 (±23.64)
1,scenario_1_B_75_unique,3.85 (±0.90),6.31 (±0.97),N/A,496.08 (±0.00),68.79 (±11.88)
2,scenario_1_B_40_unique,2.44 (±0.52),2.89 (±0.68),N/A,424.91 (±0.00),28.08 (±4.96)
3,scenario_1_B_20_unique,2.70 (±0.73),8.58 (±1.57),N/A,640.10 (±0.00),77.17 (±27.09)
4,scenario_1_B,2.04 (±0.52),7.45 (±1.51),N/A,758.98 (±0.00),71.24 (±16.79)
5,bpi2020_2processes_massive_share,N/A,3.48 (±0.58),N/A,271.92 (±0.00),27.13 (±3.60)


In [16]:
print(results_df.to_latex())

\begin{tabular}{lllllll}
\toprule
 & Scenario & LSTM_PYTORCH & TRANSFORMER & LEVEL3_XGB & PGT & TIMEAWARE \\
\midrule
0 & scenario_1_A & 2.79 (±0.97) & 2.40 (±0.47) & N/A & 490.02 (±0.00) & 73.38 (±23.64) \\
1 & scenario_1_B_75_unique & 3.85 (±0.90) & 6.31 (±0.97) & N/A & 496.08 (±0.00) & 68.79 (±11.88) \\
2 & scenario_1_B_40_unique & 2.44 (±0.52) & 2.89 (±0.68) & N/A & 424.91 (±0.00) & 28.08 (±4.96) \\
3 & scenario_1_B_20_unique & 2.70 (±0.73) & 8.58 (±1.57) & N/A & 640.10 (±0.00) & 77.17 (±27.09) \\
4 & scenario_1_B & 2.04 (±0.52) & 7.45 (±1.51) & N/A & 758.98 (±0.00) & 71.24 (±16.79) \\
5 & bpi2020_2processes_massive_share & N/A & 3.48 (±0.58) & N/A & 271.92 (±0.00) & 27.13 (±3.60) \\
\bottomrule
\end{tabular}



In [17]:
import pandas as pd
import numpy as np
from scipy import stats

results = []


models = ['LSTM_PYTORCH', 'TRANSFORMER', 'LEVEL3_XGB','PGT', 'TIMEAWARE']
scenarios = ['scenario_1_A', 'scenario_1_B_75_unique',  'scenario_1_B_40_unique', 'scenario_1_B_20_unique', 'scenario_1_B','bpi2020_2processes_massive_share']

for scenario in scenarios:
    row = {'Scenario': scenario}
    
    for model in models:
        try:
            df = pd.read_csv(f'results/{model}_{scenario}.csv')[['test_mae']]
            mae_mean = df['test_mae'].mean()
            mae_std = df['test_mae'].std()
            n = len(df)
            
            ci = stats.t.ppf(0.975, df=n-1) * (mae_std / np.sqrt(n)) if n > 1 else 0.0
            
            value = f"{mae_mean:.2f} (±{ci:.2f})"
            row[model] = value
        
        except Exception as e:
            row[model] = 'N/A'
    
    results.append(row)

results_df = pd.DataFrame(results)
results_df

,Scenario,LSTM_PYTORCH,TRANSFORMER,LEVEL3_XGB,PGT,TIMEAWARE
0,scenario_1_A,11.79 (±2.85),9.30 (±0.14),9.00 (±0.00),8.53 (±0.00),8.96 (±0.28)
1,scenario_1_B_75_unique,16.51 (±0.23),9.45 (±0.27),9.12 (±0.00),9.05 (±0.00),8.62 (±0.08)
2,scenario_1_B_40_unique,69.98 (±0.35),19.33 (±0.40),16.10 (±0.00),15.42 (±0.00),15.59 (±0.21)
3,scenario_1_B_20_unique,183.86 (±0.62),114.99 (±2.63),121.00 (±0.00),118.30 (±0.00),102.18 (±3.64)
4,scenario_1_B,77.65 (±82.24),46.85 (±0.52),49.60 (±0.00),47.41 (±0.00),45.03 (±0.36)
5,bpi2020_2processes_massive_share,N/A,2.50 (±0.01),2.82 (±0.00),2.45 (±0.00),2.49 (±0.00)


In [18]:
import pandas as pd
import numpy as np
from scipy import stats

results = []


models = ['LSTM_PYTORCH', 'TRANSFORMER',  'LEVEL3_XGB','PGT', 'TIMEAWARE']
scenarios = ['scenario_1_A_GEN', 'scenario_1_B_75_unique_GEN',  'scenario_1_B_40_unique_GEN', 'scenario_1_B_20_unique_GEN', 'scenario_1_B_GEN','bpi2020_2processes_massive_share_GEN']

for scenario in scenarios:
    row = {'Scenario': scenario}
    
    for model in models:
        try:
            df = pd.read_csv(f'results/{model}_{scenario}.csv')[['test_mae']]
            mae_mean = df['test_mae'].mean()
            mae_std = df['test_mae'].std()
            n = len(df)
            
            ci = stats.t.ppf(0.975, df=n-1) * (mae_std / np.sqrt(n)) if n > 1 else 0.0
            
            value = f"{mae_mean:.2f} (±{ci:.2f})"
            row[model] = value
        
        except Exception as e:
            row[model] = 'N/A'
    
    results.append(row)

results_df = pd.DataFrame(results)
results_df

,Scenario,LSTM_PYTORCH,TRANSFORMER,LEVEL3_XGB,PGT,TIMEAWARE
0,scenario_1_A_GEN,24.45 (±17.71),8.53 (±0.26),10.11 (±0.00),9.12 (±0.00),7.74 (±0.35)
1,scenario_1_B_75_unique_GEN,19.22 (±0.03),10.95 (±0.71),16.00 (±0.00),8.66 (±0.00),8.44 (±0.14)
2,scenario_1_B_40_unique_GEN,60.96 (±0.06),26.98 (±0.87),20.42 (±0.00),22.24 (±0.00),20.34 (±0.34)
3,scenario_1_B_20_unique_GEN,155.54 (±0.21),122.45 (±4.51),123.98 (±0.00),112.11 (±0.00),90.51 (±4.14)
4,scenario_1_B_GEN,83.94 (±67.83),50.15 (±0.59),45.35 (±0.00),52.48 (±0.00),44.71 (±0.49)
5,bpi2020_2processes_massive_share_GEN,N/A,1.77 (±0.07),2.49 (±0.00),2.35 (±0.00),1.98 (±0.06)


In [13]:
import pandas as pd
import numpy as np
from scipy import stats

results = []


models = [ 'TRANSFORMER',  'LEVEL3_XGB','PGT', 'TIMEAWARE']# 'LSTM',
logs = ["BPIC15_1", "BPIC15_2", "BPIC15_3", "BPIC15_4", "BPI_Challenge_2012", "BPIC20_DomesticDeclarations", "BPIC20_InternationalDeclarations"]# "BPI_Challenge_2013I" "env_permit", "HelpDesk","BPI_Challenge_2013C",, "Hospital", "Traffic_Fines","BPIC15_5"
scenarios = ["STD"]# "GEN", "TEST_SAMERES70_PAR"]#"TEST_SAMERES40_PAR", 

def get_scenario_names(logs, scenarios):
    log_names=[]
    for log in logs:
        for scenario in scenarios:
            log_names.append(f"{log}_{scenario}")
    return log_names

scenarios = get_scenario_names(logs, scenarios)

for scenario in scenarios:
    row = {'Scenario': scenario}
    
    for model in models:
        try:
            df = pd.read_csv(f'results/{model}_{scenario}.csv')[['test_mae']]
            mae_mean = df['test_mae'].mean()
            mae_std = df['test_mae'].std()
            n = len(df)
            
            ci = stats.t.ppf(0.975, df=n-1) * (mae_std / np.sqrt(n)) if n > 1 else 0.0
            
            value = f"{mae_mean:.2f} (±{ci:.2f})"
            row[model] = value
        
        except Exception as e:
            row[model] = 'N/A'
    
    results.append(row)

results_df = pd.DataFrame(results)
results_df.head(50)

,Scenario,TRANSFORMER,LEVEL3_XGB,PGT,TIMEAWARE
0,BPIC15_1_STD,727.54 (±0.00),856.39 (±0.00),545.53 (±0.00),548.38 (±0.00)
1,BPIC15_2_STD,1403.74 (±0.00),1603.12 (±0.00),1896.58 (±0.00),1334.27 (±0.00)
2,BPIC15_3_STD,365.53 (±0.00),508.54 (±0.00),362.89 (±0.00),386.43 (±0.00)
3,BPIC15_4_STD,2129.45 (±0.00),1943.40 (±0.00),1489.11 (±0.00),1915.63 (±0.00)
4,BPI_Challenge_2012_STD,142.10 (±0.00),164.99 (±0.00),141.72 (±0.00),147.45 (±0.00)
5,BPIC20_DomesticDeclarations_STD,89.79 (±0.00),100.29 (±0.00),91.18 (±0.00),90.01 (±0.00)
6,BPIC20_InternationalDeclarations_STD,314.28 (±0.00),412.01 (±0.00),318.22 (±0.00),313.39 (±0.00)


In [14]:
import pandas as pd
import numpy as np
from scipy import stats

results = []


models = [ 'TRANSFORMER',  'LEVEL3_XGB','PGT', 'TIMEAWARE']# 'LSTM',
logs = ["BPIC15_1", "BPIC15_2", "BPIC15_3", "BPIC15_4", "BPI_Challenge_2012", "BPIC20_DomesticDeclarations", "BPIC20_InternationalDeclarations"]# "BPI_Challenge_2013I" "env_permit", "HelpDesk","BPI_Challenge_2013C",, "Hospital", "Traffic_Fines","BPIC15_5"
scenarios = ["GEN"]# "GEN", "TEST_SAMERES70_PAR"]#"TEST_SAMERES40_PAR", 

def get_scenario_names(logs, scenarios):
    log_names=[]
    for log in logs:
        for scenario in scenarios:
            log_names.append(f"{log}_{scenario}")
    return log_names

scenarios = get_scenario_names(logs, scenarios)

for scenario in scenarios:
    row = {'Scenario': scenario}
    
    for model in models:
        try:
            df = pd.read_csv(f'results/{model}_{scenario}.csv')[['test_mae']]
            mae_mean = df['test_mae'].mean()
            mae_std = df['test_mae'].std()
            n = len(df)
            
            ci = stats.t.ppf(0.975, df=n-1) * (mae_std / np.sqrt(n)) if n > 1 else 0.0
            
            value = f"{mae_mean:.2f} (±{ci:.2f})"
            row[model] = value
        
        except Exception as e:
            row[model] = 'N/A'
    
    results.append(row)

results_df = pd.DataFrame(results)
results_df.head(50)

,Scenario,TRANSFORMER,LEVEL3_XGB,PGT,TIMEAWARE
0,BPIC15_1_GEN,1037.03 (±0.00),1114.25 (±0.00),990.01 (±0.00),947.62 (±0.00)
1,BPIC15_2_GEN,1492.72 (±0.00),1810.91 (±0.00),N/A,1549.89 (±0.00)
2,BPIC15_3_GEN,507.21 (±0.00),662.11 (±0.00),482.69 (±0.00),465.33 (±0.00)
3,BPIC15_4_GEN,1246.24 (±0.00),1356.56 (±0.00),1235.90 (±0.00),1094.81 (±0.00)
4,BPI_Challenge_2012_GEN,172.07 (±0.00),187.69 (±0.00),169.38 (±0.00),168.03 (±0.00)
5,BPIC20_DomesticDeclarations_GEN,85.82 (±0.00),100.06 (±0.00),88.72 (±0.00),85.38 (±0.00)
6,BPIC20_InternationalDeclarations_GEN,518.42 (±0.00),541.80 (±0.00),530.93 (±0.00),505.25 (±0.00)


In [15]:
import pandas as pd
import numpy as np
from scipy import stats

results = []


models = [ 'TRANSFORMER',  'LEVEL3_XGB','PGT', 'TIMEAWARE']# 'LSTM',
logs = ["BPIC15_1", "BPIC15_2", "BPIC15_3", "BPIC15_4", "BPI_Challenge_2012", "BPIC20_DomesticDeclarations", "BPIC20_InternationalDeclarations"]# "BPI_Challenge_2013I" "env_permit", "HelpDesk","BPI_Challenge_2013C",, "Hospital", "Traffic_Fines","BPIC15_5"
scenarios = ["TEST_SAMERES70_PAR"]# "GEN", "TEST_SAMERES70_PAR"]#"TEST_SAMERES40_PAR", 

def get_scenario_names(logs, scenarios):
    log_names=[]
    for log in logs:
        for scenario in scenarios:
            log_names.append(f"{log}_{scenario}")
    return log_names

scenarios = get_scenario_names(logs, scenarios)

for scenario in scenarios:
    row = {'Scenario': scenario}
    
    for model in models:
        try:
            df = pd.read_csv(f'results/{model}_{scenario}.csv')[['test_mae']]
            mae_mean = df['test_mae'].mean()
            mae_std = df['test_mae'].std()
            n = len(df)
            
            ci = stats.t.ppf(0.975, df=n-1) * (mae_std / np.sqrt(n)) if n > 1 else 0.0
            
            value = f"{mae_mean:.2f} (±{ci:.2f})"
            row[model] = value
        
        except Exception as e:
            row[model] = 'N/A'
    
    results.append(row)

results_df = pd.DataFrame(results)
results_df.head(50)

,Scenario,TRANSFORMER,LEVEL3_XGB,PGT,TIMEAWARE
0,BPIC15_1_TEST_SAMERES70_PAR,1032.85 (±0.00),844.27 (±0.00),546.36 (±0.00),595.22 (±0.00)
1,BPIC15_2_TEST_SAMERES70_PAR,1327.00 (±0.00),1585.29 (±0.00),1867.55 (±0.00),1022.36 (±0.00)
2,BPIC15_3_TEST_SAMERES70_PAR,334.46 (±0.00),486.03 (±0.00),341.89 (±0.00),309.72 (±0.00)
3,BPIC15_4_TEST_SAMERES70_PAR,1663.45 (±0.00),1943.38 (±0.00),2117.27 (±0.00),1533.18 (±0.00)
4,BPI_Challenge_2012_TEST_SAMERES70_PAR,139.21 (±0.00),165.39 (±0.00),137.78 (±0.00),134.19 (±0.00)
5,BPIC20_DomesticDeclarations_TEST_SAMERES70_PAR,89.90 (±0.00),100.26 (±0.00),92.50 (±0.00),89.99 (±0.00)
6,BPIC20_InternationalDeclarations_TEST_SAMERES7...,331.09 (±0.00),412.04 (±0.00),345.28 (±0.00),311.93 (±0.00)


In [17]:
import pandas as pd
import numpy as np
from scipy import stats

results = []


models = [ 'TRANSFORMER',  'LEVEL3_XGB','PGT', 'TIMEAWARE']# 'LSTM',
logs = ["BPIC15_1", "BPIC15_2", "BPIC15_3", "BPIC15_4", "BPI_Challenge_2012", "BPIC20_DomesticDeclarations", "BPIC20_InternationalDeclarations"]# "BPI_Challenge_2013I" "env_permit", "HelpDesk","BPI_Challenge_2013C",, "Hospital", "Traffic_Fines","BPIC15_5"
scenarios = ["TEST_SAMERES70_PAR"]# "GEN", "TEST_SAMERES70_PAR"]#"TEST_SAMERES40_PAR", 

def get_scenario_names(logs, scenarios):
    log_names=[]
    for log in logs:
        for scenario in scenarios:
            log_names.append(f"{log}_{scenario}")
    return log_names

scenarios = get_scenario_names(logs, scenarios)

for scenario in scenarios:
    row = {'Scenario': scenario}
    
    for model in models:
        try:
            df = pd.read_csv(f'results/{model}_{scenario}.csv')[['training_time_sec']]
            df['training_time_min'] = df['training_time_sec'] / 60 
            mae_mean = df['training_time_min'].mean()
            mae_std = df['training_time_min'].std()
            n = len(df)
            
            ci = stats.t.ppf(0.975, df=n-1) * (mae_std / np.sqrt(n)) if n > 1 else 0.0
            
            value = f"{mae_mean:.2f} (±{ci:.2f})"
            row[model] = value
        
        except Exception as e:
            row[model] = 'N/A'
    
    results.append(row)

results_df = pd.DataFrame(results)
results_df.head(50)

,Scenario,TRANSFORMER,LEVEL3_XGB,PGT,TIMEAWARE
0,BPIC15_1_TEST_SAMERES70_PAR,1.81 (±0.00),0.02 (±0.00),934.61 (±0.00),58.25 (±0.00)
1,BPIC15_2_TEST_SAMERES70_PAR,1.37 (±0.00),0.03 (±0.00),2076.44 (±0.00),42.04 (±0.00)
2,BPIC15_3_TEST_SAMERES70_PAR,1.55 (±0.00),0.03 (±0.00),1391.93 (±0.00),70.56 (±0.00)
3,BPIC15_4_TEST_SAMERES70_PAR,1.75 (±0.00),0.02 (±0.00),1106.32 (±0.00),41.94 (±0.00)
4,BPI_Challenge_2012_TEST_SAMERES70_PAR,13.50 (±0.00),0.04 (±0.00),3926.36 (±0.00),1083.83 (±0.00)
5,BPIC20_DomesticDeclarations_TEST_SAMERES70_PAR,3.18 (±0.00),0.03 (±0.00),669.36 (±0.00),26.22 (±0.00)
6,BPIC20_InternationalDeclarations_TEST_SAMERES7...,1.80 (±0.00),0.03 (±0.00),1238.66 (±0.00),87.62 (±0.00)


# Statistical tests

In [4]:
from framework.utils import test_mae_difference

scenarios = ['scenario_1_A', 'scenario_1_B']
gen_names = ['scenario_1_A_GEN', 'scenario_1_B_GEN']
models = ['TRANSFORMER', 'MODEL4']
for scenario in scenarios+gen_names:
    print('################### ', scenario, ' ###################')
    df_T = pd.read_csv(f'results/TRANSFORMER_{scenario}.csv')[['test_mae','training_time_sec']]
    df_M = pd.read_csv(f'results/MODEL4_{scenario}.csv')[['test_mae','training_time_sec']]
    stat, p = test_mae_difference(df_T, df_M, paired=True)
    print(f"t-statistic: {stat:.4f}, p-value: {p:.4f}")


###################  scenario_1_A  ###################
t-statistic: 0.4248, p-value: 0.6810
###################  scenario_1_B  ###################
t-statistic: 8.3192, p-value: 0.0000
###################  scenario_1_A_GEN  ###################
t-statistic: 1.7633, p-value: 0.1117
###################  scenario_1_B_GEN  ###################
t-statistic: 8.6918, p-value: 0.0000


In [7]:
from framework.utils import test_mae_difference

scenarios = ['scenario_1_A', 'scenario_1_B_75_unique', 'scenario_1_B_40_unique', 'scenario_1_B_20_unique', 'scenario_1_B']
for scenario in scenarios:
    print('################### ', scenario, ' ###################')
    df_T = pd.read_csv(f'results/TRANSFORMER_{scenario}.csv')[['test_mae','training_time_sec']]
    df_M = pd.read_csv(f'results/MODEL4_{scenario}.csv')[['test_mae','training_time_sec']]
    stat, p = test_mae_difference(df_T, df_M, paired=True)
    print(f"t-statistic: {stat:.4f}, p-value: {p:.4f}")

###################  scenario_1_A  ###################
t-statistic: 0.4248, p-value: 0.6810
###################  scenario_1_B_75_unique  ###################
t-statistic: 0.9499, p-value: 0.3670
###################  scenario_1_B_40_unique  ###################
t-statistic: 5.5169, p-value: 0.0004
###################  scenario_1_B_20_unique  ###################
t-statistic: 14.2912, p-value: 0.0000
###################  scenario_1_B  ###################
t-statistic: 8.3192, p-value: 0.0000


In [ ]:
from framework.utils import test_mae_difference

scenarios = ['scenario_1_B_75_unique']
gen_names = ['scenario_1_B_75_unique_GEN']
for scenario in scenarios+gen_names:
    print('################### ', scenario, ' ###################')
    df_T = pd.read_csv(f'results/TRANSFORMER_{scenario}.csv')[['test_mae','training_time_sec']]
    df_M = pd.read_csv(f'results/MODEL4_{scenario}.csv')[['test_mae','training_time_sec']]
    stat, p = test_mae_difference(df_T, df_M, paired=True)
    print(f"t-statistic: {stat:.4f}, p-value: {p:.4f}")

###################  scenario_1_B_75_unique  ###################
t-statistic: 0.9499, p-value: 0.3670
###################  scenario_1_B_75_unique_GEN  ###################
t-statistic: 1.3683, p-value: 0.2044
